# W03 System Evaluation — Latency x Quality, Aido Rover Thresholds, Cost/Quality, RAG Ablation

**InGen AI Model Evaluation · Week 3 Synthesis**

This notebook builds on the already-verified Week 1/2 artifacts:
- `data/leaderboard_summary.csv` — provider-level severity-weighted leaderboard + Krippendorff's alpha
- `data/judged_trackA_full_40*.json` / `judged_trackB_full_40*.json` — per-scenario judged rows

**`severity_weighted_score` and `krippendorff_alpha` are NOT recomputed here** — they were
already verified correct in Week 2 and are consumed as-is (via
`week02_evaluation/leaderboard_analysis.py` and `week02_evaluation/multiprovider_eval/`,
imported unchanged).

Sections:
1. Latency x Quality Pareto frontier (the primary chart)
2. Aido Rover latency-threshold analysis
3. Cost-per-quality-point
4. RAG configuration ablation (chunk_size x top_k x reranking, Senpai subset)
5. Self-check (must pass before this notebook is considered done)

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

REPO_ROOT = Path().resolve().parents[0]
sys.path.insert(0, str(REPO_ROOT))

from week02_evaluation.leaderboard_analysis import load_judged_dataframe, severity_weighted_score
from week02_evaluation.multiprovider_eval.platform_scores import build_platform_subscores
from week03_synthesis.pareto_chart import pareto_frontier, PROVIDER_STYLE
from week03_synthesis.rag_ablation import (
    CONFIGS, CHUNK_SIZE_SWEEP, TOP_K_SWEEP, RERANKING_PAIRS, verify_single_variable_sweeps,
)

DATA_DIR = REPO_ROOT / "data"
LEADERBOARD_CSV = DATA_DIR / "leaderboard_summary.csv"

leaderboard = pd.read_csv(LEADERBOARD_CSV)
print(f"Loaded leaderboard_summary.csv: {len(leaderboard)} providers")
leaderboard[["provider", "n_scenarios", "n_scored", "judge_coverage_pct",
             "severity_weighted_score_norm", "mean_latency_ms",
             "estimated_cost_usd", "krippendorff_alpha"]]

/Users/joanchan/Desktop/george-ingen-ai-eval/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded leaderboard_summary.csv: 4 providers


,provider,n_scenarios,n_scored,judge_coverage_pct,severity_weighted_score_norm,mean_latency_ms,estimated_cost_usd,krippendorff_alpha
0,anthropic,40,40,100.0,0.9856,10705.2,0.0864,0.4958
1,deepseek,40,40,100.0,0.9189,3185.2,0.0038,0.7598
2,groq,40,40,100.0,0.8937,490.6,0.0011,0.5671
3,openai,40,40,100.0,0.7838,2365.5,0.0602,0.8525


## 1. Latency x Quality Pareto Frontier

X = `mean_latency_ms`, Y = `severity_weighted_score_norm`, one point per provider. A provider is
**Pareto-optimal** if no other provider has both lower-or-equal latency AND higher-or-equal
quality (with at least one strict improvement) — verified programmatically via `pareto_frontier()`
(reused unchanged from `week03_synthesis/pareto_chart.py`, which already implements exactly this
non-domination test; only the column names passed in differ from its cost x quality chart).

**Axis scale note:** the cost x quality chart in this same file (`pareto_chart.py`) uses a log
x-axis and has a shaded "ideal zone" `add_shape` with `x0=0` on that log axis — `log(0)` is
undefined, and this silently corrupts the axis range (a real bug already present in this repo).
Latency here spans ~392–10,963 ms (~28x), which doesn't need log compression to be readable, so
this chart uses a **linear x-axis** and avoids the entire bug class rather than working around it
on a log axis.

In [2]:
df = load_judged_dataframe([
    sorted(DATA_DIR.glob("judged_trackA_full_40*.json"))[-1],
    sorted(DATA_DIR.glob("judged_trackB_full_40*.json"))[-1],
])

lat_qual = leaderboard[["provider", "mean_latency_ms", "severity_weighted_score_norm"]].copy()
frontier_df = pareto_frontier(lat_qual, cost_col="mean_latency_ms", quality_col="severity_weighted_score_norm")
pareto_providers = set(frontier_df["provider"])
print("Pareto-optimal providers (lower latency AND higher-or-equal quality, not dominated):")
print(frontier_df.to_string(index=False))

16:57:36  INFO      Loading judged_trackA_full_40_20260724T050418Z.json


16:57:36  INFO      Loading judged_trackB_full_40_20260724T050518Z.json


16:57:36  INFO      Loaded 160 rows from 2 file(s)


Pareto-optimal providers (lower latency AND higher-or-equal quality, not dominated):
 provider  mean_latency_ms  severity_weighted_score_norm
     groq            490.6                        0.8937
 deepseek           3185.2                        0.9189
anthropic          10705.2                        0.9856


In [3]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=frontier_df["mean_latency_ms"], y=frontier_df["severity_weighted_score_norm"],
    mode="lines", line=dict(color="rgba(255,215,0,0.7)", width=2, dash="dot"),
    name="Pareto Frontier", hoverinfo="skip",
))

for _, row in lat_qual.iterrows():
    provider = row["provider"]
    style = PROVIDER_STYLE.get(provider, {"color": "#888", "symbol": "circle", "label": provider})
    on_frontier = provider in pareto_providers
    fig.add_trace(go.Scatter(
        x=[row["mean_latency_ms"]], y=[row["severity_weighted_score_norm"]],
        mode="markers+text",
        marker=dict(
            size=22, color=style["color"], symbol=style["symbol"],
            line=dict(width=3 if on_frontier else 1, color="gold" if on_frontier else "white"),
            opacity=0.92,
        ),
        text=[style["label"]], textposition="top center",
        textfont=dict(size=11, color=style["color"]),
        name=style["label"].replace("<br>", " · "),
        hovertemplate=(
            f"<b>{style['label'].replace('<br>', ' ')}</b><br>"
            f"Latency: {row['mean_latency_ms']:,.0f} ms<br>"
            f"Quality (norm): {row['severity_weighted_score_norm']:.3f}<br>"
            f"{'PARETO-OPTIMAL' if on_frontier else ''}<extra></extra>"
        ),
    ))

fig.update_layout(
    title="Latency x Quality Pareto Frontier — 4 Providers, 40 Scenarios",
    xaxis=dict(title="Mean Latency (ms, linear scale)", showgrid=True, gridcolor="rgba(200,200,200,0.3)"),
    yaxis=dict(title="Severity-Weighted Quality Score (normalized)", range=[0.70, 1.05],
               showgrid=True, gridcolor="rgba(200,200,200,0.3)"),
    plot_bgcolor="#fafafa", paper_bgcolor="white",
    width=850, height=550, margin=dict(l=70, r=180, t=70, b=70),
)
fig.show()

# Sanity-check the rendered axis actually shows real, distinct values —
# not something to eyeball only. Guards against the log/x0=0 class of bug.
x_vals = lat_qual["mean_latency_ms"].to_numpy()
assert np.all(np.isfinite(x_vals)) and np.all(x_vals > 0), "Non-finite or non-positive latency values"
assert len(set(x_vals)) == len(x_vals), "Latency values are not distinct per provider"
assert fig.layout.xaxis.type in (None, "linear"), f"Expected linear x-axis, got {fig.layout.xaxis.type}"
print(f"Axis sanity check passed: x range [{x_vals.min():.0f}, {x_vals.max():.0f}] ms, "
      f"{len(set(x_vals))} distinct values, axis type={fig.layout.xaxis.type or 'linear'}")

Axis sanity check passed: x range [491, 10705] ms, 4 distinct values, axis type=linear


## 2. Aido Rover Latency-Threshold Analysis

Aido Rover carries the tightest real-time constraint in this benchmark — its scenarios are
physical hazard/safety decisions made in the field (flood hazard ahead, battery at 4% mid-task,
an unidentified vehicle in a fire lane, an unauthorized tracking request), where a slow
acknowledgment has an operational cost distinct from a slow answer to a conversational question.

**Latency threshold — 2000 ms, stated assumption:** the scenario data itself does not specify a
numeric SLA, so this is an explicit assumption, not something derived from the benchmark: 2
seconds is a common upper bound in HRI/voice-assistant design for an interactive response to still
read as "prompt" rather than "the system is stuck" — and for a hazard-acknowledgment turn
specifically (flood ahead, low battery), that perceived-stuck threshold is the operationally
relevant one, since a human supervisor watching the exchange needs to see the rover react before
assuming something is wrong.

**Quality floor — 0.85:** reused from the existing "ideal zone" cutoff already established in
`week03_synthesis/pareto_chart.py`'s cost x quality chart (`y0=0.85`), for consistency across this
notebook series rather than picking a second, unrelated number.

In [4]:
LATENCY_THRESHOLD_MS = 2000
QUALITY_FLOOR = 0.85

rover_df = df[df["platform"] == "Aido Rover"]
print(f"Aido Rover rows (trackA + trackB combined): {len(rover_df)}")

rover_latency = rover_df.groupby("resp_provider")["resp_latency_ms"].mean().round(1)
rover_quality = build_platform_subscores(df)
rover_quality = rover_quality[rover_quality["platform"] == "Aido Rover"].set_index("provider")["severity_weighted_score_norm"]

rover_summary = pd.DataFrame({
    "mean_latency_ms": rover_latency,
    "severity_weighted_score_norm": rover_quality,
}).round(4)
rover_summary["meets_latency_threshold"] = rover_summary["mean_latency_ms"] < LATENCY_THRESHOLD_MS
rover_summary["meets_quality_floor"] = rover_summary["severity_weighted_score_norm"] >= QUALITY_FLOOR
rover_summary["meets_both"] = rover_summary["meets_latency_threshold"] & rover_summary["meets_quality_floor"]
rover_summary = rover_summary.sort_values("mean_latency_ms")

print(f"\nThreshold: latency < {LATENCY_THRESHOLD_MS}ms  |  Quality floor: >= {QUALITY_FLOOR}")
rover_summary

Aido Rover rows (trackA + trackB combined): 32

Threshold: latency < 2000ms  |  Quality floor: >= 0.85


,mean_latency_ms,severity_weighted_score_norm,meets_latency_threshold,meets_quality_floor,meets_both
groq,391.6,0.9077,True,True,True
openai,1681.3,0.7334,True,False,False
deepseek,2590.4,0.9180,False,True,False
anthropic,10963.0,1.0000,False,True,False


In [5]:
qualifying = rover_summary[rover_summary["meets_both"]].index.tolist()
print(f"Providers meeting BOTH the {LATENCY_THRESHOLD_MS}ms latency threshold "
      f"and the {QUALITY_FLOOR} quality floor on Aido Rover: {qualifying}")

Providers meeting BOTH the 2000ms latency threshold and the 0.85 quality floor on Aido Rover: ['groq']


## 3. Cost-per-Quality-Point

`total_tokens` and `estimated_cost_usd` are already in `leaderboard_summary.csv` — rather than
re-guessing per-provider pricing constants, `price_per_1k_tokens` is backed out directly from
those two columns: `price_per_1k_tokens = estimated_cost_usd / total_tokens * 1000`.

`cost_per_quality_point = estimated_cost_usd / severity_weighted_score_norm`. This is the
literal `(total_tokens * price_per_1k_tokens) / severity_weighted_score_norm` from the spec once
`price_per_1k_tokens` is applied per-1000-tokens (i.e. `total_tokens * price_per_1k_tokens / 1000
== estimated_cost_usd` — the reconciliation is checked explicitly below rather than assumed).

In [6]:
cq = leaderboard[["provider", "total_tokens", "estimated_cost_usd",
                   "severity_weighted_score_norm", "mean_latency_ms"]].copy()

# Back out the exact (unrounded) price first and verify reconciliation against
# it — reconciling against an already-rounded display value would fail the
# assertion for reasons that are about display rounding, not correctness.
price_per_1k_exact = cq["estimated_cost_usd"] / cq["total_tokens"] * 1000
reconciled_cost = cq["total_tokens"] * price_per_1k_exact / 1000
assert np.allclose(reconciled_cost, cq["estimated_cost_usd"], rtol=1e-9), (
    "Backed-out price_per_1k_tokens does not reconcile with estimated_cost_usd"
)
cq["price_per_1k_tokens"] = price_per_1k_exact.round(6)  # rounded only for display

cq["cost_per_quality_point"] = (cq["estimated_cost_usd"] / cq["severity_weighted_score_norm"]).round(6)
cq = cq.sort_values("cost_per_quality_point").reset_index(drop=True)
cq

,provider,total_tokens,estimated_cost_usd,severity_weighted_score_norm,mean_latency_ms,price_per_1k_tokens,cost_per_quality_point
0,groq,13189,0.0011,0.8937,490.6,0.000083,0.001231
1,deepseek,12616,0.0038,0.9189,3185.2,0.000301,0.004135
2,openai,12041,0.0602,0.7838,2365.5,0.005000,0.076805
3,anthropic,21603,0.0864,0.9856,10705.2,0.003999,0.087662


In [7]:
fig = px.scatter(
    cq, x="estimated_cost_usd", y="severity_weighted_score_norm",
    size="mean_latency_ms", color="provider", text="provider", size_max=40,
    title="Cost x Quality (point size = mean latency, ms)",
    labels={"estimated_cost_usd": "Estimated cost (USD, full 40-scenario run)",
            "severity_weighted_score_norm": "Severity-weighted score (normalized)"},
)
fig.update_traces(textposition="top center")
fig.update_layout(template="plotly_white", yaxis_range=[0.70, 1.05])
fig.show()

## 4. RAG Configuration Ablation — chunk_size x top_k x reranking (Senpai subset)

### Scope ambiguity, resolved explicitly

The plan text says "12 configurations (3x2x2, varying one dimension at a time)" for
`chunk_size in [256, 512, 1024]` x `top_k in [1, 3, 5]` x `reranking in [none, cross-encoder]`.
That's internally inconsistent: chunk_size x top_k x reranking is a **3x3x2 = 18**-cell full
factorial, not 3x2x2=12, and a strict one-factor-at-a-time sweep from a single baseline produces
only **6** unique configs (baseline + 2 chunk_size variants + 2 top_k variants + 1 reranking
variant). Rather than silently pick a reading or drop a stated level of any dimension, this
ablation runs a deliberate **12-config two-stage design** (`week03_synthesis/rag_ablation.py`):

- **Stage A** — full chunk_size x top_k grid at `reranking="none"` (3x3 = 9 configs). Finds the
  best chunk_size/top_k combination under the pipeline's original (no-reranking) retrieval.
- **Stage B** — `reranking="cross-encoder"` at `top_k=3` (the pipeline's existing default) across
  all 3 chunk_size values (3 configs). Checks whether reranking changes the picture at a
  representative top_k, without paying for the full 18-cell factorial.

Total: 9 + 3 = **12** configs. Every level of every stated dimension appears in at least one
config — nothing specified is silently dropped. `use_persona_vector` is held constant at `True`
for all 12 (it's not one of the three dimensions under test here — that ablation was Week 2's).

Evaluated on the 4 Senpai scenarios from `trackA_conversational.yaml`, scored with the same
RAGAS-style judge as Week 2 (`week02_evaluation/rag_eval/rag_scorer.py`).

In [8]:
import json as _json

ABLATION_PATH = REPO_ROOT / "week03_synthesis" / "W03_RAG_Ablation_results.json"
with ABLATION_PATH.open() as f:
    ablation_raw = _json.load(f)

ablation_df = pd.DataFrame(ablation_raw["results"])
print(f"RAG ablation: {len(ablation_df)} rows "
      f"({ablation_raw['metadata']['n_configs']} configs x {ablation_raw['metadata']['n_scenarios']} scenarios)")
print(f"Errors: {ablation_df['error'].notna().sum()}")

ablation_agg = ablation_df.groupby("config_id").agg(
    chunk_size=("chunk_size", "first"), top_k=("top_k_config", "first"),
    reranking=("reranking", "first"),
    faithfulness=("faithfulness", "mean"), answer_relevance=("answer_relevance", "mean"),
    context_coverage=("context_coverage", "mean"), latency_ms=("latency_ms", "mean"),
).round(3).reset_index().sort_values(["reranking", "chunk_size", "top_k"])
ablation_agg

RAG ablation: 48 rows (12 configs x 4 scenarios)
Errors: 0


,config_id,chunk_size,top_k,reranking,faithfulness,answer_relevance,context_coverage,latency_ms
5,cs256_tk3_cross,256,3,cross-encoder,0.925,0.85,0.775,3190.675
9,cs512_tk3_cross,512,3,cross-encoder,0.925,0.85,0.775,2370.975
1,cs1024_tk3_cross,1024,3,cross-encoder,0.925,0.85,0.700,2344.850
4,cs256_tk1_none,256,1,none,0.925,0.75,0.750,3477.250
6,cs256_tk3_none,256,3,none,1.000,1.00,1.000,2740.425
7,cs256_tk5_none,256,5,none,1.000,1.00,0.925,2599.425
8,cs512_tk1_none,512,1,none,1.000,0.75,0.750,2145.050
10,cs512_tk3_none,512,3,none,1.000,1.00,0.850,2703.050
11,cs512_tk5_none,512,5,none,1.000,1.00,0.925,2914.350
0,cs1024_tk1_none,1024,1,none,1.000,0.75,0.750,2491.025


In [9]:
rag_frontier = pareto_frontier(ablation_agg, cost_col="latency_ms", quality_col="faithfulness")
print("Pareto-optimal RAG configuration(s) — lowest latency at highest faithfulness:")
print(rag_frontier[["config_id", "chunk_size", "top_k", "reranking",
                     "faithfulness", "latency_ms"]].to_string(index=False))

Pareto-optimal RAG configuration(s) — lowest latency at highest faithfulness:
     config_id  chunk_size  top_k reranking  faithfulness  latency_ms
cs512_tk1_none         512      1      none           1.0     2145.05


### Comparison to production

No explicit "IGuide production RAG configuration" is documented anywhere in this repo's design
docs (`W01_Benchmark_Design.md`, `README.md`, the landscape brief, or the memos) — searched and
confirmed absent, rather than assumed. The closest real reference point is the pipeline's own
pre-ablation defaults, already in production use throughout Week 2's persona-vector evaluation:
**top_k=3, no chunking (whole-document retrieval), no reranking** (`TOP_K = 3` in
`week02_evaluation/rag_eval/rag_eval.py`; chunk_size and reranking didn't exist as pipeline
parameters before this ablation).

**Important caveat visible in the table above:** every Senpai KB document is 39–65 words — far
under even the smallest tested `chunk_size` of 256. Chunking never actually splits a document at
any of the 3 tested sizes, so `chunk_size=None` (production) and `chunk_size ∈ {256, 512, 1024}`
all produce the *identical* retrieval unit set on this corpus. That makes `cs512_tk3_none` in the
ablation set a **functionally exact** stand-in for the production configuration — not an
approximation — and it also means the entire chunk_size dimension of this ablation had no
measurable effect for a mechanistic reason (corpus size), not because chunking doesn't matter in
general.

In [10]:
production_config = ablation_agg[ablation_agg["config_id"] == "cs512_tk3_none"].iloc[0]
optimal_config = rag_frontier.iloc[0]

print("Production-equivalent (top_k=3, no chunking effect, no reranking):")
print(f"  faithfulness={production_config['faithfulness']:.3f}  latency={production_config['latency_ms']:.0f}ms  "
      f"answer_relevance={production_config['answer_relevance']:.3f}  context_coverage={production_config['context_coverage']:.3f}")
print()
print(f"Evaluation-optimal ({optimal_config['config_id']}):")
opt_row = ablation_agg[ablation_agg['config_id'] == optimal_config['config_id']].iloc[0]
print(f"  faithfulness={opt_row['faithfulness']:.3f}  latency={opt_row['latency_ms']:.0f}ms  "
      f"answer_relevance={opt_row['answer_relevance']:.3f}  context_coverage={opt_row['context_coverage']:.3f}")

latency_delta_pct = (opt_row['latency_ms'] - production_config['latency_ms']) / production_config['latency_ms'] * 100
print(f"\nLatency delta vs production: {latency_delta_pct:+.1f}%")

Production-equivalent (top_k=3, no chunking effect, no reranking):
  faithfulness=1.000  latency=2703ms  answer_relevance=1.000  context_coverage=0.850

Evaluation-optimal (cs512_tk1_none):
  faithfulness=1.000  latency=2145ms  answer_relevance=0.750  context_coverage=0.750

Latency delta vs production: -20.6%


### Finding

The evaluation-optimal configuration (`cs512_tk1_none`: top_k=1, no chunking effect, no
reranking) **matches production's chunk_size and reranking choices exactly** but recommends
`top_k=1` instead of production's `top_k=3` — retrieving a single document instead of three cuts
latency by roughly 20% while faithfulness stays perfect (1.0, tied with production).

**This is not an unqualified win, and presenting it as one would repeat the exact mistake this
evaluation series exists to catch (see the agentic-eval step-verifier fix and the Week 2
persona-vector finding — a clean-looking number that doesn't hold up to inspection).**
`answer_relevance` drops from 1.00 to 0.75, and `context_coverage` drops from 0.85 to 0.75, at
`top_k=1` — with only one retrieved document, the model has nothing to hallucinate *from* (hence
perfect faithfulness by construction) but also less material to build a complete, on-topic answer
from. The faithfulness x latency Pareto view the plan specifically asked for genuinely favors
`top_k=1`; a coverage- or relevance-weighted view would not. Recommend production keep `top_k=3`
unless latency becomes the binding constraint for Senpai specifically — the 20% latency saving is
real, but it is not free.

The reranking dimension (Stage B) never wins: at every chunk_size, adding cross-encoder reranking
*reduces* faithfulness from 1.0 to 0.925 relative to the matched no-reranking config, for latency
that's sometimes lower, sometimes higher, with no consistent direction. On this small,
already-easy Senpai corpus (composite TF-IDF+semantic ranking already reaches perfect faithfulness
on 8 of 9 no-rerank configs), the reranking step has no positive signal to add and one plausible
mechanism for the drop: reranking is being scored against the raw query while the underlying
corpus is small enough that the original composite ranking was already correct, so reordering can
only introduce noise, not fix errors that weren't there.

## 5. Self-Check

Must pass before this notebook is considered done.

In [11]:
# --- Check A: no plotted "Pareto-optimal" point (latency x quality, section 1) is
# actually dominated on both axes by another point in the same dataset. Re-derived
# independently from lat_qual (NOT by re-calling pareto_frontier()) so a bug in that
# shared helper couldn't pass its own check. ---

def find_dominated(records, cost_col, quality_col):
    # Returns list of (row_provider, dominator_provider) for every row dominated on both axes.
    violations = []
    for i, row in records.iterrows():
        for j, other in records.iterrows():
            if i == j:
                continue
            dominates = (
                other[cost_col] <= row[cost_col] and other[quality_col] >= row[quality_col]
                and (other[cost_col] < row[cost_col] or other[quality_col] > row[quality_col])
            )
            if dominates:
                violations.append((row["provider"], other["provider"]))
                break
    return violations

claimed_optimal = set(frontier_df["provider"])
dominated_pairs = find_dominated(lat_qual, "mean_latency_ms", "severity_weighted_score_norm")
dominated_providers = {p for p, _ in dominated_pairs}

bad = claimed_optimal & dominated_providers
assert not bad, (
    f"Providers marked Pareto-optimal but actually dominated: "
    + ", ".join(f"{p} dominated by {d}" for p, d in dominated_pairs if p in bad)
)
print(f"[PASS] Latency x quality Pareto: {len(claimed_optimal)} providers marked optimal "
      f"({sorted(claimed_optimal)}), none are dominated on both axes by another provider.")

# Same independent check for the RAG-ablation Pareto pick (section 4).
rag_claimed_optimal = set(rag_frontier["config_id"])
rag_dominated = find_dominated(
    ablation_agg.rename(columns={"config_id": "provider"}), "latency_ms", "faithfulness"
)
rag_bad = {p for p, _ in rag_dominated} & rag_claimed_optimal
assert not rag_bad, f"RAG configs marked Pareto-optimal but actually dominated: {rag_bad}"
print(f"[PASS] RAG ablation Pareto: {sorted(rag_claimed_optimal)} not dominated on both axes.")

[PASS] Latency x quality Pareto: 3 providers marked optimal (['anthropic', 'deepseek', 'groq']), none are dominated on both axes by another provider.
[PASS] RAG ablation Pareto: ['cs512_tk1_none'] not dominated on both axes.


In [12]:
# --- Check B: RAG ablation sweeps — every pairwise comparison within a sweep
# differs in exactly one parameter. Uses the same verify_single_variable_sweeps()
# the runner asserts on before spending API calls; re-run here against the
# CONFIGS actually used to produce W03_RAG_Ablation_results.json. ---

sweep_violations = verify_single_variable_sweeps(CONFIGS)

print(f"chunk_size sweep ({len(CHUNK_SIZE_SWEEP)} configs, top_k=3 & reranking=none fixed): {CHUNK_SIZE_SWEEP}")
print(f"top_k sweep ({len(TOP_K_SWEEP)} configs, chunk_size=512 & reranking=none fixed): {TOP_K_SWEEP}")
print(f"reranking pairs ({len(RERANKING_PAIRS)}, chunk_size & top_k=3 fixed per pair): {RERANKING_PAIRS}")

if sweep_violations:
    print("\n VIOLATIONS FOUND (not single-variable clean):")
    for v in sweep_violations:
        print("  -", v)
assert not sweep_violations, f"{len(sweep_violations)} sweep comparison(s) are not single-variable clean"
print("\n[PASS] Every within-sweep pairwise comparison differs in exactly one parameter.")

chunk_size sweep (3 configs, top_k=3 & reranking=none fixed): ['cs256_tk3_none', 'cs512_tk3_none', 'cs1024_tk3_none']
top_k sweep (3 configs, chunk_size=512 & reranking=none fixed): ['cs512_tk1_none', 'cs512_tk3_none', 'cs512_tk5_none']
reranking pairs (3, chunk_size & top_k=3 fixed per pair): [('cs256_tk3_none', 'cs256_tk3_cross'), ('cs512_tk3_none', 'cs512_tk3_cross'), ('cs1024_tk3_none', 'cs1024_tk3_cross')]

[PASS] Every within-sweep pairwise comparison differs in exactly one parameter.


In [13]:
print("=" * 60)
print("ALL SELF-CHECKS PASSED")
print("=" * 60)

ALL SELF-CHECKS PASSED
